In [1]:
# ── CELL 1: Imports and Load Flood Risk Products ─────────────────────────────
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask
from rasterio.warp import reproject, Resampling
from shapely.geometry import Point, shape
import matplotlib.pyplot as plt
import requests
import ee
import warnings
warnings.filterwarnings('ignore')

PROJECT_DIR = os.path.expanduser("~/GeoAID_Project")
DATA_DIR    = os.path.join(PROJECT_DIR, "data")
OUTPUTS_DIR = os.path.join(PROJECT_DIR, "outputs")
FIGURES_DIR = os.path.join(OUTPUTS_DIR, "figures")

ee.Initialize(project='ee-festac')

# Load flood risk tier raster (generated in NB07)
tier_path = os.path.join(OUTPUTS_DIR, 'flood_risk_tiers.tif')
with rasterio.open(tier_path) as src:
    risk_tiers     = src.read(1)
    tier_transform = src.transform
    tier_crs       = src.crs
    tier_bounds    = src.bounds

tier_labels = {0: 'NoData', 1: 'Low', 2: 'Moderate', 3: 'High', 4: 'Very High'}

print("✓ Flood risk tier raster loaded")
print(f"  Dimensions: {risk_tiers.shape}")
for t, name in tier_labels.items():
    print(f"  {name:10}: {(risk_tiers==t).sum():,} pixels")

# Load TRUE Amuwo Odofin LGA polygon (not the rectangular raster bounding box)
amuwo_odofin = ee.FeatureCollection("FAO/GAUL/2015/level2") \
                 .filter(ee.Filter.eq('ADM0_NAME', 'Nigeria')) \
                 .filter(ee.Filter.eq('ADM1_NAME', 'Lagos')) \
                 .filter(ee.Filter.stringContains('ADM2_NAME', 'Amuwo Odofin'))

lga_shape = shape(amuwo_odofin.geometry().getInfo())

lga_mask = geometry_mask(
    [lga_shape], out_shape=risk_tiers.shape,
    transform=tier_transform, invert=True
)

print(f"\n✓ True LGA polygon loaded and mask created")
print(f"  Polygon area   : {lga_shape.area:.6f} sq degrees")
print(f"  Pixels inside true LGA boundary: {lga_mask.sum():,}")
print(f"  Note: raster bounding box is ~1.71x larger than the true polygon —")
print(f"  all infrastructure and population analysis below is clipped to")
print(f"  the true polygon to avoid contamination from adjacent LGAs")

/home/deysholey/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✓ Flood risk tier raster loaded
  Dimensions: (147, 206)
  NoData    : 17,260 pixels
  Low       : 3,256 pixels
  Moderate  : 3,255 pixels
  High      : 3,255 pixels
  Very High : 3,256 pixels

✓ True LGA polygon loaded and mask created
  Polygon area   : 0.014317 sq degrees
  Pixels inside true LGA boundary: 17,685
  Note: raster bounding box is ~1.71x larger than the true polygon —
  all infrastructure and population analysis below is clipped to
  the true polygon to avoid contamination from adjacent LGAs


In [2]:
# ── CELL 2: Query OpenStreetMap via Overpass API ─────────────────────────────
overpass_endpoints = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
]
headers = {
    'Content-Type': 'text/plain; charset=utf-8',
    'User-Agent': 'GeoAID-MSc-Research/1.0 (Miva Open University; academic research)'
}

bbox = f"{tier_bounds.bottom},{tier_bounds.left},{tier_bounds.top},{tier_bounds.right}"

def query_overpass(query_body, label, timeout_query=60, timeout_http=90):
    query = f"[out:json][timeout:{timeout_query}];\n(\n  {query_body}\n);\nout center;"
    for endpoint in overpass_endpoints:
        try:
            print(f"  Trying {endpoint.split('/')[2]}...")
            response = requests.post(endpoint, data=query.encode('utf-8'),
                                      headers=headers, timeout=timeout_http)
            if response.status_code == 200:
                data = response.json()
                print(f"  ✓ {label}: {len(data['elements'])} features")
                return data['elements']
            print(f"    status {response.status_code} — trying next endpoint")
        except Exception as e:
            print(f"    error: {e} — trying next endpoint")
    print(f"  ✗ All endpoints failed for {label}")
    return []

schools_raw = query_overpass(
    f'node["amenity"="school"]({bbox});\nway["amenity"="school"]({bbox});', "schools")
health_raw = query_overpass(
    f'node["amenity"~"hospital|clinic|doctors"]({bbox});\nway["amenity"~"hospital|clinic|doctors"]({bbox});',
    "hospitals/clinics")
roads_raw = query_overpass(
    f'way["highway"~"primary|secondary|tertiary|trunk"]({bbox});', "major roads")
buildings_raw = query_overpass(
    f'way["building"="residential"]({bbox});\nway["building"="yes"]({bbox});',
    "buildings", timeout_query=90, timeout_http=120)

print(f"\nRaw OSM query results (rectangular bounding box, unclipped):")
print(f"  Schools: {len(schools_raw)} | Health: {len(health_raw)} | "
      f"Roads: {len(roads_raw)} | Buildings: {len(buildings_raw):,}")

# Cap buildings for computational tractability — representative sample
MAX_BUILDINGS = 20000
if len(buildings_raw) > MAX_BUILDINGS:
    np.random.seed(42)
    buildings_raw = list(np.random.choice(buildings_raw, size=MAX_BUILDINGS, replace=False))
    print(f"  ⚠ Buildings capped at {MAX_BUILDINGS:,} (random sample) for tractability")

  Trying overpass-api.de...
  ✓ schools: 93 features
  Trying overpass-api.de...
  ✓ hospitals/clinics: 194 features
  Trying overpass-api.de...
  ✓ major roads: 815 features
  Trying overpass-api.de...
    status 429 — trying next endpoint
  Trying overpass.kumi.systems...
    status 504 — trying next endpoint
  Trying lz4.overpass-api.de...
    status 504 — trying next endpoint
  ✗ All endpoints failed for buildings

Raw OSM query results (rectangular bounding box, unclipped):
  Schools: 93 | Health: 194 | Roads: 815 | Buildings: 0


In [3]:
# ── CELL 3: Convert OSM Elements to GeoDataFrames, Clipped to True LGA ───────
# Critical correction: the rectangular query bounding box is 1.71x larger
# than the true LGA polygon. All infrastructure is filtered here to only
# include points genuinely inside Amuwo Odofin.

def osm_to_gdf_clipped(elements, category_name, lga_polygon):
    records = []
    for elem in elements:
        if elem['type'] == 'node':
            lon, lat = elem.get('lon'), elem.get('lat')
        elif 'center' in elem:
            lon, lat = elem['center'].get('lon'), elem['center'].get('lat')
        else:
            continue
        if lon is None or lat is None:
            continue
        point = Point(lon, lat)
        if not point.within(lga_polygon):
            continue
        tags = elem.get('tags', {})
        records.append({
            'osm_id': elem.get('id'), 'name': tags.get('name', 'Unnamed'),
            'category': category_name, 'geometry': point
        })
    return gpd.GeoDataFrame(records, crs='EPSG:4326')

schools_gdf   = osm_to_gdf_clipped(schools_raw,   'school',   lga_shape)
health_gdf    = osm_to_gdf_clipped(health_raw,    'health',   lga_shape)
roads_gdf     = osm_to_gdf_clipped(roads_raw,     'road',     lga_shape)
buildings_gdf = osm_to_gdf_clipped(buildings_raw, 'building', lga_shape)

print("Infrastructure counts — raw query vs. clipped to true LGA boundary:")
print(f"  Schools   : {len(schools_raw):>6} raw → {len(schools_gdf):>6} within true LGA")
print(f"  Health    : {len(health_raw):>6} raw → {len(health_gdf):>6} within true LGA")
print(f"  Roads     : {len(roads_raw):>6} raw → {len(roads_gdf):>6} within true LGA")
print(f"  Buildings : {len(buildings_raw):>6,} raw → {len(buildings_gdf):>6,} within true LGA")

ValueError: Assigning CRS to a GeoDataFrame without a geometry column is not supported. Supply geometry using the 'geometry=' keyword argument, or by providing a DataFrame with column name 'geometry'

In [4]:
# ── DIAGNOSTIC: Check Buildings Raw Data Structure ───────────────────────────
print(f"Total buildings_raw: {len(buildings_raw)}")
print()

# Check first 3 elements structure
for i, elem in enumerate(buildings_raw[:3]):
    print(f"Element {i}: type={elem.get('type')}, "
          f"has_center={'center' in elem}, "
          f"has_lat_lon={'lat' in elem and 'lon' in elem}")

# Count how many have usable geometry
has_center = sum(1 for e in buildings_raw if 'center' in e)
has_latlon = sum(1 for e in buildings_raw if 'lat' in e and 'lon' in e)
print(f"\nElements with 'center' key : {has_center}")
print(f"Elements with lat/lon keys : {has_latlon}")

Total buildings_raw: 0


Elements with 'center' key : 0
Elements with lat/lon keys : 0


In [ ]:
# ── CELL 4: Assign Flood Risk Tier to Infrastructure ─────────────────────────

def assign_risk_tier(gdf, tier_array, transform):
    tiers = []
    for geom in gdf.geometry:
        row, col = rasterio.transform.rowcol(transform, geom.x, geom.y)
        if 0 <= row < tier_array.shape[0] and 0 <= col < tier_array.shape[1]:
            tiers.append(tier_array[row, col])
        else:
            tiers.append(0)
    gdf = gdf.copy()
    gdf['risk_tier']  = tiers
    gdf['risk_label'] = gdf['risk_tier'].map(tier_labels)
    return gdf

schools_gdf   = assign_risk_tier(schools_gdf,   risk_tiers, tier_transform)
health_gdf    = assign_risk_tier(health_gdf,    risk_tiers, tier_transform)
buildings_gdf = assign_risk_tier(buildings_gdf, risk_tiers, tier_transform)
roads_gdf     = assign_risk_tier(roads_gdf,     risk_tiers, tier_transform)

print("Infrastructure exposure by flood risk tier (within true LGA boundary):\n")
for name, gdf in [('Schools', schools_gdf), ('Health facilities', health_gdf),
                   ('Buildings (sample)', buildings_gdf), ('Major roads', roads_gdf)]:
    print(f"{name} (n={len(gdf)}):")
    counts = gdf['risk_label'].value_counts()
    for tier in ['Low', 'Moderate', 'High', 'Very High', 'NoData']:
        c = counts.get(tier, 0)
        pct = c / len(gdf) * 100 if len(gdf) > 0 else 0
        print(f"  {tier:10}: {c:5,} ({pct:5.1f}%)")
    print()

at_risk_schools = schools_gdf[schools_gdf['risk_tier'].isin([3, 4])]
at_risk_health  = health_gdf[health_gdf['risk_tier'].isin([3, 4])]
at_risk_roads   = roads_gdf[roads_gdf['risk_tier'].isin([3, 4])]

print("✓ At-risk assets (High or Very High tier):")
print(f"  Schools           : {len(at_risk_schools)} of {len(schools_gdf)}")
print(f"  Health facilities : {len(at_risk_health)} of {len(health_gdf)}")
print(f"  Major roads       : {len(at_risk_roads)} of {len(roads_gdf)}")

✓ True LGA polygon loaded
  Polygon area: 0.014317 sq degrees
  Bounding box was 1.71x larger — this is the source of the error

Infrastructure counts — before and after clipping to true LGA:
  Schools   :  93 raw →  35 within true LGA
  Health    : 192 raw → 102 within true LGA
  Roads     : 815 raw → 208 within true LGA
  Buildings : 20,000 raw →  8,162 within true LGA


In [ ]:
# ── CELL 5: Export WorldPop Population, Clipped to True LGA via GEE ──────────
worldpop = ee.ImageCollection("WorldPop/GP/100m/pop") \
             .filter(ee.Filter.eq('country', 'NGA')) \
             .filter(ee.Filter.eq('year', 2020)) \
             .first() \
             .clip(amuwo_odofin.geometry())

pop_export_task = ee.batch.Export.image.toDrive(
    image=worldpop.toFloat(), description='amuwo_odofin_worldpop_2020',
    folder='GeoAID_Project', fileNamePrefix='worldpop_2020_amuwo_odofin',
    region=amuwo_odofin.geometry(), scale=100, crs='EPSG:4326',
    maxPixels=1e9, fileFormat='GeoTIFF'
)
pop_export_task.start()
print("✓ WorldPop export submitted — download to data/ before running Cell 6")
print("  Monitor at: https://code.earthengine.google.com/tasks")

✓ Risk tiers assigned — infrastructure now correctly bounded to true LGA

Schools (n=35) by risk tier:
  Low       :     1 (  2.9%)
  Moderate  :     4 ( 11.4%)
  High      :     5 ( 14.3%)
  Very High :    11 ( 31.4%)
  NoData    :    14 ( 40.0%)

Health facilities (n=102) by risk tier:
  Low       :    12 ( 11.8%)
  Moderate  :    24 ( 23.5%)
  High      :    33 ( 32.4%)
  Very High :    22 ( 21.6%)
  NoData    :    11 ( 10.8%)

Buildings (sample) (n=8162) by risk tier:
  Low       :   472 (  5.8%)
  Moderate  : 1,757 ( 21.5%)
  High      : 2,662 ( 32.6%)
  Very High : 2,443 ( 29.9%)
  NoData    :   828 ( 10.1%)

Major roads (n=208) by risk tier:
  Low       :     5 (  2.4%)
  Moderate  :    23 ( 11.1%)
  High      :    39 ( 18.8%)
  Very High :   109 ( 52.4%)
  NoData    :    32 ( 15.4%)

✓ At-risk assets (High or Very High tier):
  Schools           : 16 of 35
  Health facilities : 55 of 102
  Major roads       : 148 of 208


In [ ]:
# ── CELL 6: Population Exposure by Flood Risk Tier ───────────────────────────
# WorldPop reports NaN (not zero) for zero-population pixels — this is
# standard WorldPop behaviour, not a data defect. NaN is converted to 0
# before summing since it represents genuine absence of recorded population.

pop_path = os.path.join(DATA_DIR, 'worldpop_2020_amuwo_odofin.tif')
with rasterio.open(pop_path) as src:
    pop_data      = src.read(1)
    pop_transform = src.transform

if pop_data.shape != risk_tiers.shape:
    pop_resampled = np.zeros(risk_tiers.shape, dtype=np.float32)
    reproject(source=pop_data, destination=pop_resampled,
              src_transform=pop_transform, src_crs='EPSG:4326',
              dst_transform=tier_transform, dst_crs='EPSG:4326',
              resampling=Resampling.sum)
    pop_data = pop_resampled

pop_data_clean   = np.nan_to_num(pop_data, nan=0.0)
pop_data_masked  = np.where(lga_mask, pop_data_clean, 0)
risk_data_masked = np.where(lga_mask, risk_tiers, 0)

print("Population exposure by flood risk tier (within true LGA boundary):\n")
total_pop = pop_data_masked.sum()
pop_by_tier = {}
for tier, name in [(1, 'Low'), (2, 'Moderate'), (3, 'High'), (4, 'Very High')]:
    tier_pop = pop_data_masked[risk_data_masked == tier].sum()
    pop_by_tier[name] = tier_pop
    pct = tier_pop / total_pop * 100 if total_pop > 0 else 0
    print(f"  {name:10}: {tier_pop:>10,.0f} people ({pct:5.1f}%)")

high_vhigh_pop = pop_data_masked[(risk_data_masked==3)|(risk_data_masked==4)].sum()
print(f"\n  Total population (true LGA)        : {total_pop:,.0f}")
print(f"  Population in High + Very High risk: {high_vhigh_pop:,.0f} "
      f"({high_vhigh_pop/total_pop*100:.1f}%)")

✓ WorldPop export submitted — clipped to true Amuwo Odofin LGA
  Monitor at: https://code.earthengine.google.com/tasks
  Once complete, download 'worldpop_2020_amuwo_odofin.tif' to
  /home/deysholey/GeoAID_Project/data and proceed to Cell 6

  Estimated total population (WorldPop 2020): 442,357


In [ ]:
# ── CELL 7: Visualise Infrastructure Exposure and Export Products ───────────
from matplotlib.colors import ListedColormap

fig, ax = plt.subplots(figsize=(11, 9))
tier_colours = ListedColormap(['#FFFFFF', '#2ECC71', '#F1C40F', '#E67E22', '#C0392B'])
display = np.where(lga_mask, risk_tiers, np.nan)
im = ax.imshow(display, cmap=tier_colours, vmin=0, vmax=4)

for gdf, marker, color, label in [
    (schools_gdf, '^', 'blue', 'Schools'),
    (health_gdf, 's', 'purple', 'Health facilities')
]:
    rows, cols = [], []
    for geom in gdf.geometry:
        r, c = rasterio.transform.rowcol(tier_transform, geom.x, geom.y)
        rows.append(r); cols.append(c)
    ax.scatter(cols, rows, marker=marker, c=color, s=40,
               edgecolors='white', linewidths=0.8, label=label, zorder=5)

ax.set_title('GeoAID Disaster Intelligence Layer\n'
             'Infrastructure Exposure — Amuwo Odofin LGA',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'infrastructure_risk_map.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# Export summary tables
schools_gdf.drop(columns='geometry').to_csv(
    os.path.join(OUTPUTS_DIR, 'schools_risk_exposure.csv'), index=False)
health_gdf.drop(columns='geometry').to_csv(
    os.path.join(OUTPUTS_DIR, 'health_risk_exposure.csv'), index=False)
roads_gdf.drop(columns='geometry').to_csv(
    os.path.join(OUTPUTS_DIR, 'roads_risk_exposure.csv'), index=False)

pop_summary = pd.DataFrame([
    {'tier': k, 'population': v, 'percent': v/total_pop*100}
    for k, v in pop_by_tier.items()
])
pop_summary.to_csv(os.path.join(OUTPUTS_DIR, 'population_exposure.csv'), index=False)

print("✓ Infrastructure risk map saved")
print("✓ Exposure tables exported: schools, health, roads, population")

✓ LGA polygon mask recreated: 17,685 pixels inside true boundary

WorldPop raster: (147, 206)
Risk tier raster: (147, 206)
✓ Population raster already matches risk tier grid — no resampling needed

Population exposure by flood risk tier (within true LGA boundary):
  Low       :        nan people (  0.0%)
  Moderate  :        nan people (  0.0%)
  High      :        nan people (  0.0%)
  Very High :        nan people (  0.0%)

  Total population (true LGA)        : nan
  Population in High + Very High risk: nan


In [ ]:
# ── CELL 8: Session Summary ───────────────────────────────────────────────────
print("=" * 60)
print("  NOTEBOOK 09 — INFRASTRUCTURE RISK ASSESSMENT: COMPLETE")
print("=" * 60)
print()
print("  Critical correction applied:")
print("  Rectangular raster bbox (1.71x larger than true LGA polygon)")
print("  was contaminating OSM queries with out-of-boundary infrastructure.")
print("  All analysis now clipped to true FAO GAUL LGA polygon.")
print()
print("  Infrastructure at risk (High + Very High tiers):")
print(f"  Schools           : {len(at_risk_schools)} of {len(schools_gdf)}")
print(f"  Health facilities : {len(at_risk_health)} of {len(health_gdf)}")
print(f"  Major roads       : {len(at_risk_roads)} of {len(roads_gdf)}")
print()
print(f"  Population exposure:")
print(f"  Total (true LGA)         : {total_pop:,.0f}")
print(f"  High + Very High risk    : {high_vhigh_pop:,.0f} "
      f"({high_vhigh_pop/total_pop*100:.1f}%)")
print()
print("  Next: NB10 — Streamlit dashboard development")
print("=" * 60)